# Menebak Berat Badan dari Catatan Harian

**Materi: Exploratory Data Analysis (EDA).**

**Apa itu EDA?** EDA adalah tahap "kenalan dulu sama data" sebelum bikin model atau menarik kesimpulan. Ibarat mau masak: cek dulu bahan apa yang ada, mana yang segar, mana yang busuk — baru mulai masak. Tanpa EDA, model gampang ngaco karena datanya belum dipahami.

**Alur EDA yang dipakai di notebook ini:**

| Tahap EDA | Bagian di notebook |
|---|---|
| 1. Kumpulkan | Bagian 1: ambil data dari Kaggle |
| 2. Pahami | Bagian 2: intip isi data |
| 3. Bersihkan | Bagian 3: buang baris yang rusak |
| 4. Visualisasi | Bagian 4–5: grafik tren & sebaran |
| 5. Uji pola | Bagian 6–9: korelasi, perbandingan, model |
| 6. Simpulkan | Kesimpulan |

Setiap judul bagian di bawah ada label tahap EDA-nya. Ikuti alurnya — itu inti dari EDA.

Analisis Python dengan **data asli dari Kaggle** — dataset *"2018 Calorie, Exercise and Weight Changes"* (chrisbow), dataset yang dipakai notebook *"My Weight Loss Regression Dataset and Analysis"*.

Sumber data: https://www.kaggle.com/datasets/chrisbow/2018-calorie-exercise-and-weight-changes

Jalankan kotak kode berurutan dari atas (klik kotaknya, tekan `Shift + Enter`). Saat pertama dijalankan, data asli otomatis diunduh dari Kaggle, jadi bisa dijalankan di mana saja termasuk Google Colab.

In [ ]:
# Kotak 0 — Siapkan peralatan
# (Di Google Colab semua ini sudah tersedia. Di laptop sendiri kalau belum ada: pip install pandas numpy matplotlib statsmodels scipy)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats

print("Siap! Peralatan sudah dimuat.")

## 1. Ambil datanya dari Kaggle  *(tahap: kumpulkan)*
Kotak ini mengunduh file `diet_data.csv` langsung dari Kaggle (tidak perlu login). Kalau file sudah ada di folder yang sama, unduhan dilewati.

In [ ]:
# Kotak 1 — Unduh data asli dari Kaggle
import os, io, urllib.request, zipfile

URL = "https://www.kaggle.com/api/v1/datasets/download/chrisbow/2018-calorie-exercise-and-weight-changes"
CSV = "diet_data.csv"

if not os.path.exists(CSV):
    print("Mengunduh data asli dari Kaggle...")
    with urllib.request.urlopen(URL, timeout=120) as r:
        blob = r.read()
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        z.extract(CSV)
    print("Berhasil! Tersimpan sebagai", CSV)
else:
    print("File sudah ada, unduhan dilewati.")

df = pd.read_csv(CSV)
print("Jumlah baris:", len(df), "| Jumlah kolom:", len(df.columns))

## 2. Intip datanya  *(tahap: pahami)*
Satu baris = satu hari. Ini arti tiap kolom pentingnya:

| Kolom | Artinya |
|---|---|
| `Date` | Tanggal pencatatan |
| `weight_oz` | Berat badan hari itu (ons) |
| `calories` | Total kalori yang dimakan hari itu |
| `cals_per_oz` | Kalori dibagi berat badan — "seberapa padat" makannya relatif ke badan |
| `five_donuts` | 1 = hari itu makan donat |
| `walk` / `run` | 1 = hari itu jalan cepat / lari |
| `wine` / `prot` | 1 = hari itu minum wine / diet protein |
| `change` | Berat naik (+) / turun (−) hari itu (ons) |

In [ ]:
# Kotak 2 — Lihat 5 hari pertama dan ringkasannya
display(df.head())
print()
print(df.describe().round(1))

## 3. Bersih-bersih dulu  *(tahap: bersihkan)*
Data mentah selalu sedikit berantakan. Tiga hal yang dirapikan (persis seperti di notebook aslinya):
1. Kolom timbangan mentah (`Stone/Pounds/Ounces`) dibuang — sudah ada versi ons-nya.
2. Kolom `cals_per_oz` ada tulisan error `#DIV/0!` dari spreadsheet — diubah jadi angka (yang error jadi kosong).
3. Baris yang masih kosong dibuang.

In [ ]:
# Kotak 3 — Bersihkan
sebelum = len(df)
df["Date"] = pd.to_datetime(df["Date"])
df["cals_per_oz"] = pd.to_numeric(df["cals_per_oz"], errors="coerce")
df = df.drop(columns=["Stone", "Pounds", "Ounces"])
df = df.dropna().reset_index(drop=True)
print(f"Baris sebelum: {sebelum} -> sesudah dibersihkan: {len(df)}")
print("Periode:", df["Date"].min().date(), "s/d", df["Date"].max().date())

Langkah khas EDA berikutnya: **peta korelasi** — melihat faktor mana saja yang bergerak bersama. Angka 1 = searah sempurna, -1 = berlawanan, 0 = tidak berhubungan.

In [ ]:
# Peta korelasi antar faktor
kolom = ["weight_oz", "calories", "cals_per_oz", "walk", "run",
         "five_donuts", "wine", "prot", "change"]
kor = df[kolom].corr()
plt.figure(figsize=(7.5, 5.5))
plt.imshow(kor, cmap="Greens", vmin=-1, vmax=1)
plt.xticks(range(len(kolom)), kolom, rotation=30, ha="right")
plt.yticks(range(len(kolom)), kolom)
plt.colorbar(label="korelasi")
plt.title("Faktor mana yang bergerak bersama?")
for i in range(len(kolom)):
    for j in range(len(kolom)):
        plt.text(j, i, f"{kor.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
plt.tight_layout(); plt.show()

## 4. Lihat gambarnya dulu  *(tahap: visualisasi)*
Sebelum menghitung apa-apa, lihat dulu bentuk datanya.

In [ ]:
# Kotak 4 — Tren berat badan dari waktu ke waktu
plt.figure(figsize=(8, 4))
plt.plot(df["Date"], df["weight_oz"], color="#0F766E", linewidth=2)
plt.xlabel("Tanggal"); plt.ylabel("Berat badan (ons)")
plt.title("Berat badan menurun selama ±5 bulan")
plt.grid(alpha=0.3); plt.show()

Beratnya menurun — naik-turun harian itu wajar, trennya yang penting. Gambar kedua: hubungan kalori harian dengan perubahan berat.

In [ ]:
# Kotak 5 — Scatter: kalori vs perubahan berat (hari normal, <= 3500 kalori)
df_ringan = df[df["calories"] <= 3500].copy()  # hari "kalap" dibahas terpisah di Bagian 8
plt.figure(figsize=(8, 4))
plt.scatter(df_ringan["calories"], df_ringan["change"], color="#14B8A6", alpha=0.5, s=25)
xs = np.linspace(1500, 3500, 50)
plt.plot(xs, -75.91 + 0.0274 * xs, color="#0F766E", linewidth=2.5)
plt.xlabel("Kalori yang dimakan hari itu"); plt.ylabel("Perubahan berat hari itu (ons)")
plt.title("Makin banyak kalori, makin naik beratnya")
plt.grid(alpha=0.3); plt.show()

## 5. Model pertama: seberapa besar pengaruh kalori?  *(tahap: uji pola)*
Kita cari garis lurus yang paling cocok: `perubahan berat = a + b × kalori`. Angka `b` = **angka pengaruh** — tiap 1 kalori ≈ berapa ons.

In [ ]:
# Kotak 6 — Cari angka pengaruh kalori
hasil = smf.ols("change ~ calories", data=df_ringan).fit()
b = hasil.params["calories"]
print(f"Jumlah hari normal: {len(df_ringan)}")
print(f"Angka pengaruh kalori: {b:.4f} ons per kalori")
print(f"Artinya: tiap +100 kalori  ->  {b*100:+.1f} ons ({b*100*28.35:+.0f} gram) perubahan berat.")

## 6. Soal donat  *(tahap: uji pola)*
Rata-rata hari makan donat beratnya turun sedikit lebih banyak (-13,2 vs -10,3 ons). Apa artinya? Kita cek dua cara: dilihat sekilas, lalu dibandingkan dengan benar.

In [ ]:
# Kotak 7a — Bandingan sekilas
rata = df_ringan.groupby("five_donuts")["change"].mean()
print("Rata-rata perubahan berat:")
print(f"  Hari donat      : {rata[1]:+.1f} ons")
print(f"  Hari tanpa donat: {rata[0]:+.1f} ons")
p = stats.ttest_ind(df_ringan[df_ringan.five_donuts == 1]["change"],
                    df_ringan[df_ringan.five_donuts == 0]["change"]).pvalue
print(f"Uji statistik: p = {p:.2f} ->", "TERBUKTI beda" if p < 0.05 else "BELUM TERBUKTI (bisa jadi kebetulan)")
print("Catatan: hari donat kebetulan juga lebih banyak jalan cepatnya (0,87 vs 0,75) — ada faktor pengganggu!")

In [ ]:
# Kotak 7b — Bandingan adil: samakan dulu kalorinya, baru nilai donatnya
hasil = smf.ols("change ~ calories + five_donuts", data=df_ringan).fit()
efek, p = hasil.params["five_donuts"], hasil.pvalues["five_donuts"]
print(f"Efek donat setelah kalori disamakan: {efek:+.1f} ons, p = {p:.2f} ->",
      "TERBUKTI" if p < 0.05 else "TIDAK TERBUKTI berpengaruh")
print()
print("Kesimpulannya: tidak ada bukti donat berpengaruh. Yang jelas berpengaruh adalah total kalori.")

## 7. Lari vs jalan cepat  *(tahap: uji pola)*
Hari berlari vs tidak berlari — bedanya jelas? Bagaimana dengan jalan cepat? (Keduanya diuji sambil menyamakan kalori.)

In [ ]:
# Kotak 8 — Efek olahraga
hasil = smf.ols("change ~ calories + walk + run", data=df_ringan).fit()
for nama in ["walk", "run"]:
    efek, p = hasil.params[nama], hasil.pvalues[nama]
    status = "TERBUKTI membantu" if p < 0.05 else "tidak terbukti"
    print(f"{nama:5s}: {efek:+.1f} ons  (p = {p:.3f}) -> {status}")
print()
print("Menarik: jalan cepat tidak terbukti di data, tapi penulis aslinya tetap mempertahankannya")
print("karena pengalamannya — angka bukan satu-satunya penentu keputusan.")

## 8. Hari "kalap makan"  *(tahap: uji pola)*
Hari makan > 5.000 kalori ada 22 hari. Berat melonjak... tapi apa yang terjadi 1–3 hari sesudahnya?

In [ ]:
# Kotak 9 — Pola sesudah hari kalap
kalap = df.index[df["calories"] > 5000].tolist()
print(f"Jumlah hari kalap: {len(kalap)}")
rata_lag = []
for lag, label in [(0, "hari kalap"), (1, "1 hari sesudah"), (2, "2 hari sesudah"), (3, "3 hari sesudah")]:
    vals = [df.loc[i + lag, "change"] for i in kalap if i + lag < len(df)]
    rata_lag.append(np.mean(vals))
    print(f"  Rata-rata {label:15s}: {np.mean(vals):+.1f} ons")
plt.figure(figsize=(6, 3.5))
plt.bar(["hari kalap", "+1 hari", "+2 hari", "+3 hari"], rata_lag, color="#0F766E", alpha=0.85)
plt.ylabel("Rata-rata perubahan berat (ons)")
plt.title('Berat melonjak, lalu luruh dalam 2-3 hari')
plt.grid(alpha=0.3, axis="y"); plt.show()

## 9. Model final: seberapa cocok?  *(tahap: uji pola)*
Gabungkan semuanya — `cals_per_oz` (padatnya makan relatif ke badan) + lari + efek hari kalap dan 3 hari sesudahnya — lalu lihat **skor kecocokannya** (0–100%): seberapa bagus model menjelaskan naik-turunnya data.

In [ ]:
# Kotak 10 — Model final
df["binge"] = (df["calories"] > 5000).astype(int)
df["b1"] = df["binge"].shift(1).fillna(0)  # 1 hari sesudah kalap
df["b2"] = df["binge"].shift(2).fillna(0)  # 2 hari sesudah kalap
df["b3"] = df["binge"].shift(3).fillna(0)  # 3 hari sesudah kalap
fitur = ["cals_per_oz", "run", "binge", "b1", "b2", "b3"]
mfinal = smf.ols("change ~ " + " + ".join(fitur), data=df).fit()
print(f"Skor kecocokan model: {mfinal.rsquared:.0%}")
print("-> Sekitar 73% naik-turun berat harian bisa dijelaskan. Tinggi untuk data kebiasaan manusia!")

## Kesimpulan  *(tahap: simpulkan)*
- **Yang penting total kalori**, bukan jenis makanannya (donat, wine, protein: tidak terbukti).
- **Jangan tertipu pola permukaan** — samakan dulu faktor lainnya sebelum menyimpulkan.
- **Lari terbukti membantu** di data ini (±7,9 ons / ±224 gram).
- **Hari kalap tidak menghancurkan segalanya** — beratnya luruh dalam 2–3 hari.
- **Model sederhana tapi jujur** menjelaskan ±73% data.

*Catatan: ini data satu orang selama lima bulan, bukan studi klinis. Jangan digeneralisasi jadi nasihat diet — ambil pelajarannya tentang cara membaca data dengan jujur.*

**Sumber data:** dataset Kaggle *"2018 Calorie, Exercise and Weight Changes"* (chrisbow) —
https://www.kaggle.com/datasets/chrisbow/2018-calorie-exercise-and-weight-changes —
dataset yang dipakai notebook *"My Weight Loss Regression Dataset and Analysis"*. Terima kasih kepada pencatat datanya!